# המעבד — הקלטה → כתוביות מסונכרנות (SRT)

**לפני הריצה הראשונה (פעם אחת):** ודא שסביבת הריצה עם GPU —
תפריט Runtime ← Change runtime type ← **T4 GPU** ← Save.

**שימוש (מהטלפון):**
1. ודא שההקלטה שמורה ב-Google Drive.
2. מלא את תא הפרמטרים שמתחת.
3. ‏Runtime ← **Run all**. בפעם הראשונה תתבקש לאשר גישה ל-Drive — אשר.
4. בסוף: קובץ ה-SRT נשמר ב-Drive באותה תיקייה של ההקלטה, ומוצג כאן לבדיקה.

**אחרי זה:** CapCut Web ← ייבוא ה-SRT ← עיצוב ← Cloud ← אפליקציית CapCut בטלפון ← פרסום.

In [ ]:
#@title תא 1 — פרמטרים { display-mode: "form" }
#@markdown שם הספר (אפשר בעברית):
BOOK = "בראשית"  #@param ["בראשית", "שמות", "ויקרא", "במדבר", "דברים"]
#@markdown הפרק שבו ההקלטה מתחילה:
CHAPTER = 1        #@param {type:"integer"}
#@markdown טווח פסוקים — השאר 0 כדי לקחת את הפרק כולו:
VERSE_START = 0    #@param {type:"integer"}
VERSE_END = 0      #@param {type:"integer"}
#@markdown אם העלייה נגמרת בפרק אחר — מלא גם את אלה (אחרת השאר 0):
CHAPTER_END = 0    #@param {type:"integer"}
#@markdown נתיב ההקלטה ב-Drive (התיקיות תחת MyDrive):
AUDIO_PATH = "/content/drive/MyDrive/kriat/recording.m4a"  #@param {type:"string"}
#@markdown מקסימום מילים במקטע כתובית:
MAX_WORDS = 4      #@param {type:"integer"}

BRANCH = "main"
print(f"ספר {BOOK}, פרק {CHAPTER}" +
      (f", פסוקים {VERSE_START}-{VERSE_END or ''}" if VERSE_START else " (כולו)") +
      (f" עד פרק {CHAPTER_END}" if CHAPTER_END else ""))
print("הקלטה:", AUDIO_PATH)

In [ ]:
#@title תא 2 — התקנה חד-פעמית (כדקה בריצה הראשונה)
import os, subprocess, sys, torch

if not torch.cuda.is_available():
    print("⚠️ אין GPU! ‏Runtime ← Change runtime type ← T4 GPU, ואז Run all מחדש.")
    print("   (ירוץ גם על CPU, אבל לאט משמעותית)")
else:
    print("✅ GPU:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "uroman", "faster-whisper", "silero-vad"], check=True)

if os.path.exists("Kriat-Hatora"):
    subprocess.run(["git", "-C", "Kriat-Hatora", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/michaelnahmias1/Kriat-Hatora.git"], check=True)
sys.path.insert(0, "Kriat-Hatora/src")
print("✅ הקוד מוכן")

from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists(AUDIO_PATH):
    raise SystemExit(f"❌ הקובץ לא נמצא: {AUDIO_PATH}\n"
                     "בדוק את הנתיב בתא 1 (רגיש לאותיות גדולות/קטנות)")
print("✅ ההקלטה נמצאה")

In [ ]:
#@title תא 3 — עיבוד היברידי: טקסט ← תמלול ← עוגנים ← יישור ← SRT
from pathlib import Path
from aligner.hybrid import run_hybrid

audio = Path(AUDIO_PATH)
out_srt = str(audio.with_suffix(".srt"))

run_hybrid(book=BOOK,
           chapter=CHAPTER,
           audio_path=AUDIO_PATH,
           out_srt_path=out_srt,
           verse_start=VERSE_START or None,
           chapter_end=CHAPTER_END or None,
           verse_end=VERSE_END or None,
           max_words=MAX_WORDS)

print(f"\n📄 ה-SRT נשמר ב-Drive: {out_srt}")

In [ ]:
#@title תא 4 — בדיקת עיניים, דוח איכות והורדה ישירה
print(Path(out_srt).read_text(encoding="utf-8-sig")[:2500])
print("…\n")

report_path = Path(out_srt).with_suffix(".report.txt")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
print("בדוק: האם הזמנים מתקדמים בקצב הגיוני? האם הטקסט מנוקד ומוטעם כמו שצריך?")

try:
    from google.colab import files
    files.download(out_srt)  # גם הורדה ישירה לטלפון, נוסף לעותק ב-Drive
except Exception:
    pass

## אם משהו נראה לא מסונכרן

- **קרא קודם את דוח האיכות (תא 4)** — הוא מפרט בדיוק אילו מקטעים חשודים ולמה.
- **"כיסוי התאמה" נמוך בשלב 5?** התמלול התקשה בקריאה הזו — הסנכרון נשען יותר על
  היישור הכפוי. בדוק שהטווח בתא 1 תואם בדיוק את מה שנקרא בהקלטה.
- **הכתוביות זזות באיחור קבוע** — ההקלטה כנראה כוללת קטע לפני תחילת הקריאה
  (הכרזה, שקט). העוגנים אמורים לספוג את זה; אם לא — חתוך את תחילת ההקלטה.
- **מקטעים בודדים עקומים** — גרור אותם ידנית ב-CapCut לפי רשימת הדוח; זו בדיוק
  הזרימה המתוכננת.
- הקלטות ארוכות נתמכות (החלונות מפצלים אוטומטית) — אין יותר מגבלת 6 דקות.